In [1]:
import os
import deeptile
import matplotlib.pyplot as plt
import numpy as np
from deeptile.extensions import stitch
from tifffile import TiffFile
import tifffile
import dask.array as da
import utils
import skimage as ski
import importlib
import extract_features
import pandas as pd
from collections import defaultdict
from extract_features import features_basic, foci_features, feature_table, neighbor_measurements
from scipy import ndimage as ndi
import time

In [2]:
root_local = "/Users/hannahbolen/Desktop/image_analysis/"
img_name_local = "o8p_day7_s12.ome.tif"
img_path_local = os.path.join(root_local, img_name_local)
t0_local = time.time()
img_local = da.from_zarr(tifffile.imread(img_path_local, aszarr=True))
dt_nuclei_local = deeptile.load(img_local[0,20826:20826+6144,20826:20826+6144])
#dt_foci_local = deeptile.load(img_local[1,20826:20826+6144,20826:20826+6144])
print("open time:", time.time() - t0_local)

open time: 0.22637701034545898


In [2]:
root = "/Users/hannahbolen/server_mount/tiff_images/"
img_name = "o8p_day7_s12.ome.tif"
img_path = os.path.join(root, img_name)
t0 = time.time()
img = da.from_zarr(tifffile.imread(img_path, aszarr=True))
dt_nuclei = deeptile.load(img[0,20826:20826+6144,20826:20826+6144])
#dt_foci = deeptile.load(img[1,20826:20826+6144,20826:20826+6144])
print("open time:", time.time() - t0)

open time: 0.5578739643096924


In [4]:
# Configure
overlap = (0.1, 0.1)
tile_size = (2048, 2048)
# Get nuceli tiles
tiles_nuclei = dt_nuclei.get_tiles(tile_size, overlap)
tiles_nuclei = tiles_nuclei.pad()
# Get foci tiles
# tiles_foci = dt_foci.get_tiles(tile_size, overlap)
# tiles_foci = tiles_foci.pad()

In [5]:
# Segment tiles and stitch
t0 = time.time()
model_parameters = {'gpu': True, 'model_type': 'nuclei'}
eval_parameters = {'diameter': 60}
cellpose = utils.cellpose_segmentation(model_parameters, eval_parameters)

masks_nuclei = cellpose(tiles_nuclei)
t2 = time.time()
mask_nuclei = stitch.stitch_masks(masks_nuclei)
t3 = time.time()
print("cellpose w remote images run time:", time.time() - t0, "tile size:", tiles_nuclei[0,0].shape, "num tiles:", tiles_nuclei.size)

2026-03-17 20:31:38,888 [INFO] WRITING LOG OUTPUT TO /Users/hannahbolen/.cellpose/run.log
2026-03-17 20:31:38,889 [INFO] 
cellpose version: 	4.0.8 
platform:       	darwin 
python version: 	3.11.14 
torch version:  	2.9.1
2026-03-17 20:31:38,889 [WARNING] model_type argument is not used in v4.0.1+. Ignoring this argument...
2026-03-17 20:31:38,905 [INFO] ** TORCH MPS version installed and working. **
2026-03-17 20:31:38,906 [INFO] >>>> using GPU (MPS)
2026-03-17 20:31:39,536 [INFO] >>>> loading model /Users/hannahbolen/.cellpose/models/cpsam
2026-03-17 20:31:42,208 [INFO] processing grayscale image with (2048, 2048) HW
2026-03-17 20:31:50,165 [INFO] processing grayscale image with (2048, 2048) HW
2026-03-17 20:31:56,996 [INFO] processing grayscale image with (2048, 2048) HW
2026-03-17 20:32:03,560 [INFO] processing grayscale image with (2048, 2048) HW
2026-03-17 20:32:11,915 [INFO] processing grayscale image with (2048, 2048) HW
2026-03-17 20:32:19,435 [INFO] processing grayscale image

In [4]:
# Configure
overlap = (0.1, 0.1)
tile_size = (2048, 2048)
# Get nuceli tiles
tiles_nuclei_local = dt_nuclei_local.get_tiles(tile_size, overlap)
tiles_nuclei_local = tiles_nuclei_local.pad()
# # Get foci tiles
# tiles_foci = dt_foci.get_tiles(tile_size, overlap)
# tiles_foci = tiles_foci.pad()

In [9]:
# Segment tiles and stitch
tlocal = time.time()
model_parameters = {'gpu': True, 'model_type': 'nuclei'}
eval_parameters = {'diameter': 60}
cellpose = utils.cellpose_segmentation(model_parameters, eval_parameters)

masks_nuclei_local = cellpose(tiles_nuclei_local)
t2 = time.time()
mask_nuclei_local = stitch.stitch_masks(masks_nuclei_local)
t3 = time.time()
print("cellpose w local images run time:", time.time() - tlocal, "tile size:", tiles_nuclei_local[0,0].shape, "num tiles:", tiles_nuclei_local.size)

2026-03-17 20:02:38,377 [INFO] WRITING LOG OUTPUT TO /Users/hannahbolen/.cellpose/run.log
2026-03-17 20:02:38,378 [INFO] 
cellpose version: 	4.0.8 
platform:       	darwin 
python version: 	3.11.14 
torch version:  	2.9.1
2026-03-17 20:02:38,378 [WARNING] model_type argument is not used in v4.0.1+. Ignoring this argument...
2026-03-17 20:02:38,381 [INFO] ** TORCH MPS version installed and working. **
2026-03-17 20:02:38,381 [INFO] >>>> using GPU (MPS)
2026-03-17 20:02:39,063 [INFO] >>>> loading model /Users/hannahbolen/.cellpose/models/cpsam
2026-03-17 20:02:39,444 [INFO] processing grayscale image with (2048, 2048) HW
2026-03-17 20:02:46,126 [INFO] processing grayscale image with (2048, 2048) HW
2026-03-17 20:02:52,570 [INFO] processing grayscale image with (2048, 2048) HW
2026-03-17 20:02:59,071 [INFO] processing grayscale image with (2048, 2048) HW
2026-03-17 20:03:05,538 [INFO] processing grayscale image with (2048, 2048) HW
2026-03-17 20:03:12,106 [INFO] processing grayscale image

In [10]:
ts = time.time()
tifffile.imwrite(os.path.join(root_local, "".join([img_name_local.split(".")[0], "_nuclei_mask"])), mask_nuclei_local.astype("uint16"))
print("save mask locally time:", time.time()-ts)

save mask locally time: 0.020341157913208008


In [7]:
root_server = "/Users/hannahbolen/server_mount/masks/"
tss = time.time()
tifffile.imwrite(os.path.join(root_server, "".join([img_name.split(".")[0], "_nuclei_mask2"])), mask_nuclei.astype("uint16"))
print("save mask to server time:", time.time()-tss)

save mask to server time: 19.105276346206665


In [14]:
del tiles_nuclei_local, mask_nuclei_local, masks_nuclei_local, img_local, dt_nuclei_local

In [ ]:
t0 = time()
# Segment tiles and stitch
model_parameters = {'gpu': True, 'model_type': 'nuclei'}
eval_parameters = {'diameter': 60}
cellpose = utils.cellpose_segmentation(model_parameters, eval_parameters)

masks_nuclei2048 = cellpose(tiles_nuclei2048)

mask_nuclei2048 = stitch.stitch_masks(masks_nuclei2048)

print(f"""
Total time for {tiles_nuclei2048.size} 2048x2048 tiles: {round((finish_stitching_2048-start_2048)//60)} min {round((finish_stitching_2048-start_2048)%60)} sec.
Time for cellpose: {round((finish_cellpose_2048-start_2048)//60)} min {round((finish_cellpose_2048-start_2048)%60)} sec.
Time for stitching: {round((finish_stitching_2048-finish_cellpose_2048)//60)} min {round((finish_stitching_2048-finish_cellpose_2048)%60)} sec.
""")